# CyberHackAI phishing URL — V11 campaign-aware standalone

Новая версия построена **с нуля** и не зависит от предыдущих notebook, PKL или промежуточных CSV.

Нужны только:

- `train.csv`
- `test.csv`
- `sample_submission.csv`

В этой версии специально **не повторяется прежняя схема** с одной random validation и обычным char-TF-IDF + SGD как главным решением.

Основная идея:

1. выделить доменные и URL-кампании;
2. проверять качество не только случайным CV, но и `StratifiedGroupKFold` по root-domain;
3. отдельно моделировать:
   - hostname,
   - path/query,
   - нормализованный URL,
   - статистику доменных кампаний;
4. использовать две разные линейные модели и одну табличную модель;
5. объединять их через OOF logistic stack;
6. выбирать threshold только по OOF;
7. сформировать два воспроизводимых submission.

Цель — повысить устойчивость на private leaderboard, а не просто получить красивую метрику на лёгком random split.


In [1]:
# 0. Environment
import os, sys, re, math, gc, time, pickle, warnings
from urllib.parse import urlsplit

import numpy as np
import pandas as pd

from scipy.sparse import hstack, csr_matrix
from sklearn.model_selection import StratifiedKFold, StratifiedGroupKFold
from sklearn.feature_extraction.text import HashingVectorizer, TfidfTransformer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC
from sklearn.calibration import CalibratedClassifierCV
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.metrics import (
    f1_score, precision_score, recall_score, roc_auc_score,
    precision_recall_curve
)

warnings.filterwarnings("ignore")

SEED = 2026
N_SPLITS = 5
FAST_MODE = False

print("Python:", sys.version.split()[0])
print("pandas:", pd.__version__)
print("CPU:", os.cpu_count())


Python: 3.14.0
pandas: 2.3.3
CPU: 10


In [2]:
# 1. Load data
train = pd.read_csv("train.csv")
test = pd.read_csv("test.csv")
sample = pd.read_csv("sample_submission.csv")

# поддерживаем обе версии названия target из описания соревнования
if "Predicted" in train.columns:
    TARGET = "Predicted"
elif "result" in train.columns:
    TARGET = "result"
else:
    raise ValueError("В train.csv не найден target: ожидался Predicted или result")

assert "url" in train.columns
assert "url" in test.columns
assert "Id" in test.columns
assert {"Id", "Predicted"}.issubset(sample.columns)

train["url"] = train["url"].fillna("").astype(str)
test["url"] = test["url"].fillna("").astype(str)
train[TARGET] = train[TARGET].astype(int)

print("train:", train.shape)
print("test:", test.shape)
print("sample:", sample.shape)
print("target:", TARGET)
print(train[TARGET].value_counts(normalize=True).sort_index())


train: (185910, 3)
test: (46478, 2)
sample: (46478, 2)
target: Predicted
Predicted
0    0.498279
1    0.501721
Name: proportion, dtype: float64


In [ ]:
# 2. URL parser and campaign keys
def normalize_digits(s):
    s = re.sub(r"[0-9a-f]{12,}", "{hex}", s.lower())
    s = re.sub(r"\d{3,}", "{num}", s)
    return s

def parse_url(u):
    raw = str(u).strip()
    candidate = raw if "://" in raw else "http://" + raw
    try:
        p = urlsplit(candidate)
        host = (p.hostname or "").lower().strip(".")
        path = (p.path or "").lower()
        query = (p.query or "").lower()
        scheme = (p.scheme or "").lower()
    except Exception:
        host = path = query = scheme = ""

    labels = [x for x in host.split(".") if x]
    root = ".".join(labels[-2:]) if len(labels) >= 2 else host
    tld = labels[-1] if labels else ""
    subdomain = ".".join(labels[:-2]) if len(labels) > 2 else ""

    path_tokens = [x for x in path.split("/") if x]
    first_path = path_tokens[0] if path_tokens else ""

    norm_path = normalize_digits(path)
    norm_query = normalize_digits(query)
    normalized = host + norm_path + ("?" + norm_query if norm_query else "")

    campaign = root + "|" + (first_path[:50] if first_path else "{root}")
    template = root + "|" + norm_path[:140]

    return host, root, tld, subdomain, path, query, first_path, normalized, campaign, template

COLS = [
    "host","root","tld","subdomain","path","query",
    "first_path","normalized","campaign","template"
]

parts_train = pd.DataFrame([parse_url(x) for x in train["url"]], columns=COLS)
parts_test = pd.DataFrame([parse_url(x) for x in test["url"]], columns=COLS)

display(parts_train.head())


In [ ]:
# 3. Deduplicate label conflicts only
tmp = pd.DataFrame({
    "url": train["url"],
    "y": train[TARGET]
})

nuniq = tmp.groupby("url")["y"].nunique()
conflicts = set(nuniq[nuniq > 1].index)

keep = ~train["url"].isin(conflicts)
train_clean = train.loc[keep].drop_duplicates("url", keep="first").reset_index(drop=True)
y = train_clean[TARGET].to_numpy(np.int8)

parts_train = pd.DataFrame([parse_url(x) for x in train_clean["url"]], columns=COLS)

exact_lookup = (
    train.groupby("url")[TARGET]
    .agg(lambda s: int(s.iloc[0]) if s.nunique() == 1 else np.nan)
    .dropna()
    .astype(int)
    .to_dict()
)

print("conflicting urls:", len(conflicts))
print("clean train:", train_clean.shape)
print("positive rate:", round(float(y.mean()), 6))


## 4. Новый EDA: кампании и переносимость

Вместо повторения прежнего EDA по длинам URL здесь исследуется то, что непосредственно влияет на leaderboard:

- насколько много повторяющихся root-domain;
- насколько часто домен является почти одноклассовым;
- сколько test URL принадлежат уже знакомым кампаниям;
- насколько random split потенциально легче, чем split по доменным группам.


In [ ]:
# 4. Campaign-aware EDA
eda = pd.DataFrame({
    "y": y,
    "host": parts_train["host"],
    "root": parts_train["root"],
    "campaign": parts_train["campaign"],
    "template": parts_train["template"],
})

for col in ["host","root","campaign","template"]:
    st = eda.groupby(col)["y"].agg(["count","mean","nunique"])
    repeated_rows = int(st.loc[st["count"] > 1, "count"].sum())
    pure_groups = float((st["nunique"] == 1).mean())
    print(
        f"{col:10s}",
        "groups=", len(st),
        "repeated-row-share=", round(repeated_rows/len(eda), 4),
        "pure-group-share=", round(pure_groups, 4)
    )

print("\nKnown structures in test:")
for col in ["host","root","campaign","template"]:
    known = parts_test[col].isin(set(parts_train[col]))
    print(col, round(float(known.mean()), 4))


In [ ]:
# 5. Structural numeric features
SUSP = [
    "login","signin","verify","secure","account","password","payment",
    "billing","update","confirm","recover","unlock","wallet","invoice",
    "support","auth","bank","bonus","gift","free","redirect"
]

def entropy(s):
    if not s:
        return 0.0
    _, c = np.unique(list(s), return_counts=True)
    p = c/c.sum()
    return float(-(p*np.log2(p)).sum())

def build_numeric(urls, parts):
    urls = pd.Series(urls).astype(str).reset_index(drop=True)
    out = pd.DataFrame(index=np.arange(len(urls)))
    low = urls.str.lower()

    out["url_len"] = urls.str.len()
    out["host_len"] = parts["host"].str.len().to_numpy()
    out["path_len"] = parts["path"].str.len().to_numpy()
    out["query_len"] = parts["query"].str.len().to_numpy()
    out["dots"] = urls.str.count(r"\.")
    out["dashes"] = urls.str.count("-")
    out["digits"] = urls.str.count(r"\d")
    out["ats"] = urls.str.count("@")
    out["percents"] = urls.str.count("%")
    out["equals"] = urls.str.count("=")
    out["slashes"] = urls.str.count("/")
    out["subdomain_depth"] = parts["subdomain"].str.count(r"\.") + (parts["subdomain"] != "").astype(int)
    out["path_depth"] = parts["path"].str.count("/")
    out["is_ip"] = parts["host"].str.match(r"^(?:\d{1,3}\.){3}\d{1,3}$").astype(int).to_numpy()
    out["punycode"] = parts["host"].str.contains("xn--", regex=False).astype(int).to_numpy()
    out["https_token_host"] = parts["host"].str.contains("https", regex=False).astype(int).to_numpy()
    out["suspicious_hits"] = 0
    for w in SUSP:
        out["suspicious_hits"] += low.str.contains(w, regex=False).astype(int)
    out["entropy_url"] = [entropy(x) for x in urls]
    out["entropy_host"] = [entropy(x) for x in parts["host"]]
    out["entropy_path"] = [entropy(x) for x in parts["path"]]

    return out.astype(np.float32)

num_train = build_numeric(train_clean["url"], parts_train)
num_test = build_numeric(test["url"], parts_test)

display(num_train.describe().T)


In [ ]:
# 6. Frequency features — без target leakage
FREQ_COLS = ["host","root","campaign","template"]

def frequency_features(base_parts, target_parts):
    out = []
    for col in FREQ_COLS:
        vc = base_parts[col].value_counts()
        vals = np.log1p(target_parts[col].map(vc).fillna(0).to_numpy(dtype=np.float32))
        out.append(vals)
    return np.column_stack(out).astype(np.float32)

# full-train frequency features подходят для production.
freq_full_train = frequency_features(parts_train, parts_train)
freq_full_test = frequency_features(parts_train, parts_test)


In [ ]:
# 7. Stateless text representations
# Hashing avoids a global vocabulary fit and makes the whole notebook reproducible.

host_vec = HashingVectorizer(
    analyzer="char",
    ngram_range=(2,5),
    n_features=2**16,
    alternate_sign=False,
    norm=None,
    lowercase=True,
    dtype=np.float32
)

path_vec = HashingVectorizer(
    analyzer="char",
    ngram_range=(3,5),
    n_features=2**17,
    alternate_sign=False,
    norm=None,
    lowercase=True,
    dtype=np.float32
)

norm_vec = HashingVectorizer(
    analyzer="char",
    ngram_range=(3,6),
    n_features=2**18,
    alternate_sign=False,
    norm=None,
    lowercase=True,
    dtype=np.float32
)

print("Hash spaces:", 2**16, 2**17, 2**18)


In [ ]:
# 8. Fold-safe target encoding for campaign statistics
TE_COLS = ["root","campaign","template"]

def te_maps(idx):
    maps = {}
    prior = float(y[idx].mean())
    for col in TE_COLS:
        tmp = pd.DataFrame({
            "k": parts_train[col].iloc[idx].astype(str).to_numpy(),
            "y": y[idx]
        })
        st = tmp.groupby("k")["y"].agg(["sum","count"])
        alpha = 12.0
        rate = (st["sum"] + alpha*prior)/(st["count"] + alpha)
        maps[col] = (rate, st["count"], prior)
    return maps

def apply_te(maps, p):
    feats = []
    for col in TE_COLS:
        rate, count, prior = maps[col]
        s = p[col].astype(str)
        feats.append(s.map(rate).fillna(prior).to_numpy(np.float32))
        feats.append(np.log1p(s.map(count).fillna(0).to_numpy(np.float32)))
    return np.column_stack(feats).astype(np.float32)


In [ ]:
# 9. One fold trainer
BASE_NAMES = ["norm_svc","host_lr","path_lr","tabular"]

def train_fold(tr_idx, va_idx, make_test=True, seed=SEED):
    # normalized URL model — LinearSVC
    A = norm_vec.transform(parts_train["normalized"].iloc[tr_idx])
    B = norm_vec.transform(parts_train["normalized"].iloc[va_idx])
    tf_norm = TfidfTransformer(sublinear_tf=True)
    A = tf_norm.fit_transform(A)
    B = tf_norm.transform(B)

    m_norm = LinearSVC(C=0.7)
    m_norm.fit(A, y[tr_idx])
    val_norm = m_norm.decision_function(B)

    # hostname logistic
    Ah = host_vec.transform(parts_train["host"].iloc[tr_idx])
    Bh = host_vec.transform(parts_train["host"].iloc[va_idx])
    tf_h = TfidfTransformer(sublinear_tf=True)
    Ah = tf_h.fit_transform(Ah)
    Bh = tf_h.transform(Bh)

    m_host = LogisticRegression(
        C=2.0, max_iter=1500, solver="liblinear",
        random_state=seed
    )
    m_host.fit(Ah, y[tr_idx])
    val_host = m_host.predict_proba(Bh)[:,1]

    # path/query logistic
    Ap = path_vec.transform(
        (parts_train["path"].iloc[tr_idx] + "?" + parts_train["query"].iloc[tr_idx])
    )
    Bp = path_vec.transform(
        (parts_train["path"].iloc[va_idx] + "?" + parts_train["query"].iloc[va_idx])
    )
    tf_p = TfidfTransformer(sublinear_tf=True)
    Ap = tf_p.fit_transform(Ap)
    Bp = tf_p.transform(Bp)

    m_path = LogisticRegression(
        C=1.5, max_iter=1500, solver="liblinear",
        random_state=seed+1
    )
    m_path.fit(Ap, y[tr_idx])
    val_path = m_path.predict_proba(Bp)[:,1]

    # tabular = numeric + frequency + fold-safe target encoding
    freq_tr = frequency_features(parts_train.iloc[tr_idx], parts_train.iloc[tr_idx])
    freq_va = frequency_features(parts_train.iloc[tr_idx], parts_train.iloc[va_idx])
    maps = te_maps(tr_idx)
    te_tr = apply_te(maps, parts_train.iloc[tr_idx])
    te_va = apply_te(maps, parts_train.iloc[va_idx])

    Xtab_tr = np.column_stack([
        num_train.iloc[tr_idx].to_numpy(),
        freq_tr,
        te_tr
    ])
    Xtab_va = np.column_stack([
        num_train.iloc[va_idx].to_numpy(),
        freq_va,
        te_va
    ])

    m_tab = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_iter=450 if not FAST_MODE else 80,
        max_leaf_nodes=31,
        l2_regularization=3.0,
        random_state=seed+2
    )
    m_tab.fit(Xtab_tr, y[tr_idx])
    val_tab = m_tab.predict_proba(Xtab_va)[:,1]

    val = np.column_stack([val_norm, val_host, val_path, val_tab]).astype(np.float32)

    test_scores = None
    if make_test:
        T = tf_norm.transform(norm_vec.transform(parts_test["normalized"]))
        Th = tf_h.transform(host_vec.transform(parts_test["host"]))
        Tp = tf_p.transform(path_vec.transform(parts_test["path"] + "?" + parts_test["query"]))

        freq_te = frequency_features(parts_train.iloc[tr_idx], parts_test)
        te_te = apply_te(maps, parts_test)
        Xtab_te = np.column_stack([
            num_test.to_numpy(),
            freq_te,
            te_te
        ])

        test_scores = np.column_stack([
            m_norm.decision_function(T),
            m_host.predict_proba(Th)[:,1],
            m_path.predict_proba(Tp)[:,1],
            m_tab.predict_proba(Xtab_te)[:,1]
        ]).astype(np.float32)

    artifacts = {
        "tf_norm": tf_norm, "m_norm": m_norm,
        "tf_h": tf_h, "m_host": m_host,
        "tf_p": tf_p, "m_path": m_path,
        "m_tab": m_tab
    }
    return val, test_scores, artifacts


In [ ]:
# 10. Random 5-fold production OOF + test ensemble
skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

oof_random = np.zeros((len(train_clean), len(BASE_NAMES)), dtype=np.float32)
test_base = np.zeros((len(test), len(BASE_NAMES)), dtype=np.float64)
fold_artifacts = []

for fold, (tr, va) in enumerate(skf.split(train_clean, y), 1):
    t0 = time.time()
    val_sc, tst_sc, art = train_fold(tr, va, make_test=True, seed=SEED+fold)
    oof_random[va] = val_sc
    test_base += tst_sc / N_SPLITS
    fold_artifacts.append(art)
    print(f"random fold {fold}: {time.time()-t0:.1f}s")
    gc.collect()


In [ ]:
# 11. Strict grouped CV по root-domain
sgkf = StratifiedGroupKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED+99
)

oof_group = np.zeros((len(train_clean), len(BASE_NAMES)), dtype=np.float32)

for fold, (tr, va) in enumerate(
    sgkf.split(train_clean, y, groups=parts_train["root"]), 1
):
    t0 = time.time()
    val_sc, _, _ = train_fold(tr, va, make_test=False, seed=SEED+100+fold)
    oof_group[va] = val_sc
    print(f"group fold {fold}: {time.time()-t0:.1f}s")
    gc.collect()


In [ ]:
# 12. Compare base models under both validations
def best_thr(y_true, score):
    p, r, t = precision_recall_curve(y_true, score)
    f = 2*p*r/(p+r+1e-15)
    if len(t) == 0:
        return 0.5, float(f[0])
    i = int(np.nanargmax(f[:-1]))
    return float(t[i]), float(f[i])

rows = []
for j, name in enumerate(BASE_NAMES):
    tr, fr = best_thr(y, oof_random[:,j])
    tg, fg = best_thr(y, oof_group[:,j])
    rows.append({
        "model": name,
        "random_f1": fr,
        "group_f1": fg,
        "gap": fr-fg
    })

base_results = pd.DataFrame(rows).sort_values("group_f1", ascending=False)
display(base_results)


In [ ]:
# 13. OOF stack
# Для meta-модели используются только OOF base predictions.

meta_oof = np.zeros(len(train_clean), dtype=float)
meta_test = np.zeros(len(test), dtype=float)

meta_skf = StratifiedKFold(
    n_splits=N_SPLITS,
    shuffle=True,
    random_state=SEED+500
)

meta_models = []

for fold, (tr, va) in enumerate(meta_skf.split(oof_random, y), 1):
    meta = LogisticRegression(
        C=0.5,
        max_iter=3000,
        random_state=SEED+500+fold
    )
    meta.fit(oof_random[tr], y[tr])
    meta_oof[va] = meta.predict_proba(oof_random[va])[:,1]
    meta_test += meta.predict_proba(test_base)[:,1] / N_SPLITS
    meta_models.append(meta)

thr_stack, f1_stack = best_thr(y, meta_oof)
pred_stack_oof = (meta_oof >= thr_stack).astype(int)

print("Random OOF stack F1:", f1_stack)
print("precision:", precision_score(y, pred_stack_oof))
print("recall:", recall_score(y, pred_stack_oof))
print("threshold:", thr_stack)


In [ ]:
# 14. Group-robust fixed blend
# Вес выбираем исходя из grouped validation:
# модель с меньшим gap получает больший вес.

g = base_results.set_index("model")["group_f1"].reindex(BASE_NAMES).to_numpy()
g = np.maximum(g - 0.50, 0.01)
robust_weights = g / g.sum()

# z-normalization по random OOF
mu = oof_random.mean(axis=0)
sd = oof_random.std(axis=0) + 1e-9

rz = (oof_random - mu) / sd
tz = (test_base - mu) / sd
gz = (oof_group - oof_group.mean(axis=0)) / (oof_group.std(axis=0)+1e-9)

robust_oof = rz @ robust_weights
robust_test = tz @ robust_weights
robust_group = gz @ robust_weights

thr_robust, f1_robust = best_thr(y, robust_oof)
thr_group, f1_group = best_thr(y, robust_group)

print("weights:", dict(zip(BASE_NAMES, np.round(robust_weights,4))))
print("robust random F1:", f1_robust)
print("robust grouped F1:", f1_group)
print("threshold:", thr_robust)


In [ ]:
# 15. Final predictions + exact deterministic overlap
stack_pred = (meta_test >= thr_stack).astype(np.int8)
robust_pred = (robust_test >= thr_robust).astype(np.int8)

overlap = test["url"].isin(exact_lookup).to_numpy()
for i in np.flatnonzero(overlap):
    known = int(exact_lookup[test.iloc[i]["url"]])
    stack_pred[i] = known
    robust_pred[i] = known

print("exact overlap:", int(overlap.sum()))
print("stack phishing rate:", float(stack_pred.mean()))
print("robust phishing rate:", float(robust_pred.mean()))
print("disagreement:", float(np.mean(stack_pred != robust_pred)))


In [ ]:
# 16. Save submissions
sub_stack = pd.DataFrame({
    "Id": test["Id"].to_numpy(copy=True),
    "Predicted": stack_pred.astype(int)
})

sub_robust = pd.DataFrame({
    "Id": test["Id"].to_numpy(copy=True),
    "Predicted": robust_pred.astype(int)
})

for sub in [sub_stack, sub_robust]:
    assert sub.columns.tolist() == ["Id","Predicted"]
    assert len(sub) == len(test)
    assert set(sub["Predicted"].unique()).issubset({0,1})
    assert sub["Id"].reset_index(drop=True).equals(
        sample["Id"].reset_index(drop=True)
    )

sub_stack.to_csv("submission_v11_stack.csv", index=False)
sub_robust.to_csv("submission_v11_group_robust.csv", index=False)

print("saved submission_v11_stack.csv")
print("saved submission_v11_group_robust.csv")


In [ ]:
# 17. Save reproducibility bundle
bundle = {
    "version": "V11_CAMPAIGN_AWARE",
    "python": sys.version.split()[0],
    "pandas": pd.__version__,
    "base_names": BASE_NAMES,
    "stack_threshold": thr_stack,
    "robust_threshold": thr_robust,
    "robust_weights": robust_weights,
    "normalization": {"mean": mu, "std": sd},
    "meta_models": meta_models,
    "fold_artifacts": fold_artifacts,
    "exact_lookup": exact_lookup,
    "notes": (
        "Standalone campaign-aware model. "
        "Uses random 5-fold for production OOF/test ensemble and "
        "root-domain grouped 5-fold as robustness validation."
    )
}

with open("best_phishing_v11_bundle.pkl", "wb") as f:
    pickle.dump(bundle, f, protocol=pickle.HIGHEST_PROTOCOL)

print("saved best_phishing_v11_bundle.pkl")


## Что выбирать для private

`submission_v11_stack.csv` — основной вариант, если stack показывает лучший random OOF и не имеет большого разрыва с grouped quality базовых моделей.

`submission_v11_group_robust.csv` — второй вариант, специально смещённый в сторону моделей, которые лучше держатся при разбиении по root-domain.

Эти два файла должны иметь различающийся профиль ошибок и поэтому подходят как два независимых кандидата для private leaderboard.
